# Phần A đánh giá 1.000 câu trên Colab T4 (TTS VieNeu v3 Turbo + ASR)

Theo `docs/ke_hoach_sinh_1000_cau.md`; chỉ phần A (`eval1000_tts/shard_A.jsonl`, 498 câu). **Không chạy phần B.**

Chuẩn bị: tải `colab_eval_A.zip` và `colab_eval_A.zip.sha256` (từ `tts_gen/pack_colab.py` trên laptop) lên `MyDrive/asr_tts_eval/`.
Runtime → Change runtime type → **T4 GPU**.

Thứ tự:
1. Ô 1–6: cấu hình, kiểm máy, gắn Drive và giải nén, cài thư viện, tải mô hình theo revision đã ghim, hàm phụ.
   Mở phiên mới hoặc sau khi khởi động lại runtime: chạy lại ô 1–6 (đã làm rồi thì tự bỏ qua).
2. Ô 7–9: **kiểm kiểu số** (pilot100 fp16 và fp32, ASR, bảng so với bf16 trên laptop) → **DỪNG**, gửi bảng, chờ chọn.
3. Điền `DTYPE` ở ô 1, rồi ô 10 trở đi: chạy thử 24 câu, 4 câu Thái Sơn, phần A (có thử ngắt), đổi giọng, ASR, báo cáo, đóng gói.

Mọi kết quả ghi vào `/content/proj/tts_gen/runs/...`, sau mỗi bước nén và chép lên `MyDrive/asr_tts_eval/A/`.
Chạy lại một ô thì tiếp tục từ manifest, không sinh trùng câu.

In [ ]:
# Ô 1. Cấu hình
import os, json, time, subprocess, hashlib, shutil, glob, sys, tarfile, zipfile
DRIVE_DIR = "/content/drive/MyDrive/asr_tts_eval"      # chứa colab_eval_A.zip và .sha256
OUT_DRIVE = DRIVE_DIR + "/A"                             # kết quả chép lên đây
PROJ = "/content/proj"; TG = PROJ + "/tts_gen"
PIN_DIR = "/content/hf_pinned"; CT2_DIR = "/content/ct2/PhoWhisper-large-int8"
DTYPE = None   # điền "float16" hoặc "float32" SAU khi xem bảng so kiểu số (ô 9). Không trộn hai kiểu trong phần A.
# Revision đã dùng trên laptop: runs/pilot100/run_config.json (hf_revisions) và cache HF của laptop.
PINS = {
    "vieneu": ("pnnbao-ump/VieNeu-TTS-v3-Turbo", "61b85e3d937fbbacb387714180e8182823512523",
               ["config.json", "denoiser.onnx", "speaker_encoder.onnx", "update/*"]),
    "moss": ("OpenMOSS-Team/MOSS-Audio-Tokenizer-Nano", "6aa02b01e445cc585582cf0ba480bc3ea6c8dd68", None),
    "PhoWhisper-medium": ("vinai/PhoWhisper-medium", "55a7e3eb6c906de891f8f06a107754427dd3be79", None),
    "PhoWhisper-large": ("vinai/PhoWhisper-large", "b9136a44b5f2ca664bd0b8f74baecf1715f6eeeb", None),
    "wav2vec2": ("nguyenvulebinh/wav2vec2-base-vietnamese-250h", "69e9000591623e5a4fc2f502407860bcdc0de0b2", None),
}
VERSIONS = {"vieneu": "3.8.3", "transformers": "4.57.6", "ctranslate2": "4.8.3", "faster-whisper": "1.2.1"}
PIN = {k: f"{PIN_DIR}/{k}" for k in PINS}
TTS_PIN_ARGS = ["--backbone-repo", PIN["vieneu"], "--moss-tokenizer", PIN["moss"]]
PY = sys.executable
print("DTYPE =", DTYPE)

In [ ]:
# Ô 2. Kiểm GPU là T4, RAM, ổ đĩa
import psutil
print(subprocess.run(["nvidia-smi"], capture_output=True, text=True).stdout)
gpu = subprocess.run(["nvidia-smi", "--query-gpu=name,memory.total,driver_version", "--format=csv,noheader"],
                     capture_output=True, text=True).stdout.strip()
v = psutil.virtual_memory(); du = shutil.disk_usage("/content")
print(f"GPU: {gpu}\nRAM tổng {v.total / 2**30:.1f} GB, available {v.available / 2**30:.1f} GB | /content trống {du.free / 2**30:.0f} GB")
assert "T4" in gpu, "Không phải T4: Runtime > Change runtime type > T4 GPU"

In [ ]:
# Ô 3. Gắn Drive, kiểm sha256 zip, giải nén (không xóa kết quả đã có trong /content/proj)
from google.colab import drive
drive.mount("/content/drive")
os.makedirs(OUT_DRIVE, exist_ok=True)
zp = DRIVE_DIR + "/colab_eval_A.zip"

def sha256_file(p):
    h = hashlib.sha256()
    with open(p, "rb") as f:
        for b in iter(lambda: f.read(1 << 20), b""):
            h.update(b)
    return h.hexdigest()

want = open(zp + ".sha256").read().split()[0]
got = sha256_file(zp)
assert got == want, f"sha256 zip lệch: {got} != {want}"
marker = PROJ + "/.zip_sha256"
if not (os.path.exists(marker) and open(marker).read() == got):
    os.makedirs(PROJ, exist_ok=True)
    with zipfile.ZipFile(zp) as z:
        z.extractall(PROJ)
    open(marker, "w").write(got)
    print("Đã giải nén", zp)
bad = []
for line in open(TG + "/DANH_SACH.tsv", encoding="utf-8").read().splitlines()[1:]:
    rel, size, h = line.split("\t")
    if sha256_file(PROJ + "/" + rel) != h:
        bad.append(rel)
print(f"zip sha256 {got} KHỚP | tệp lệch sha256 so với DANH_SACH.tsv: {bad or 'không'}")

In [ ]:
# Ô 4. Cài thư viện (ghim), ghi phiên bản thực tế (torch là bản có sẵn của Colab)
# Nếu pip báo phải khởi động lại: Runtime > Restart session, rồi chạy lại từ ô 1.
!pip -q install "vieneu==3.8.3" "transformers==4.57.6" "ctranslate2==4.8.3" "faster-whisper==1.2.1" soxr psutil py-cpuinfo
import importlib.metadata as md, platform, torch
env = dict(gpu=gpu if "gpu" in dir() else None, python=platform.python_version(), torch=torch.__version__,
           torch_cuda=torch.version.cuda, cudnn=torch.backends.cudnn.version(), bf16=torch.cuda.is_bf16_supported(),
           packages={p: md.version(p) for p in ["vieneu", "transformers", "ctranslate2", "faster-whisper", "huggingface_hub",
                                                "tokenizers", "onnxruntime", "sea-g2p", "soxr", "numpy", "soundfile"]},
           pins={k: v[1] for k, v in PINS.items()}, ghi_luc=time.strftime("%Y-%m-%d %H:%M:%S"))
for p, ver in VERSIONS.items():
    assert env["packages"][p] == ver, f"{p} {env['packages'][p]} != {ver}"
os.makedirs(TG + "/runs", exist_ok=True)
json.dump(env, open(TG + "/runs/colab_env.json", "w"), ensure_ascii=False, indent=1)
print(json.dumps(env, ensure_ascii=False, indent=1))

In [ ]:
# Ô 5. Tải mô hình đúng revision đã ghim vào /content/hf_pinned; chuyển PhoWhisper-large sang CT2 int8 (như laptop)
from huggingface_hub import snapshot_download, HfApi
api = HfApi()
for key, (repo, rev, allow) in PINS.items():
    d = PIN[key]
    if os.path.exists(d + "/.rev") and open(d + "/.rev").read() == rev:
        continue
    names = api.list_repo_files(repo, revision=rev)
    ign = ["*.msgpack", "*.h5", "*.ot", "flax_model*", "tf_model*", "onnx/*", "onnx_int8/*", "onnx_update/*", "gguf/*"]
    if key == "PhoWhisper-large":
        ign += ["*.safetensors"]          # laptop chuyển CT2 từ pytorch_model.bin
    elif any(n.endswith(".safetensors") for n in names):
        ign += ["*.bin"]
    snapshot_download(repo, revision=rev, local_dir=d, allow_patterns=allow, ignore_patterns=ign)
    open(d + "/.rev", "w").write(rev)
    print("Tải", repo, rev[:8], "->", d)
ct2_tar = DRIVE_DIR + "/ct2_PhoWhisper-large-int8.tar"
if not os.path.exists(CT2_DIR + "/model.bin"):
    if os.path.exists(ct2_tar):
        os.makedirs(os.path.dirname(CT2_DIR), exist_ok=True)
        with tarfile.open(ct2_tar) as t:
            t.extractall(os.path.dirname(CT2_DIR))
    else:
        subprocess.run(["ct2-transformers-converter", "--model", PIN["PhoWhisper-large"], "--output_dir", CT2_DIR,
                        "--quantization", "int8_float16", "--low_cpu_mem_usage",
                        "--copy_files", "tokenizer.json", "preprocessor_config.json"], check=True)
        with tarfile.open(ct2_tar + ".tmp", "w") as t:
            t.add(CT2_DIR, arcname=os.path.basename(CT2_DIR))
        os.replace(ct2_tar + ".tmp", ct2_tar)
src = json.load(open(TG + "/.cache/ct2/PhoWhisper-large-int8/nguon.json"))
h = sha256_file(CT2_DIR + "/model.bin")
print("CT2 model.bin sha256", h, "KHỚP laptop" if h == src["model_bin_sha256"] else f"KHÁC laptop ({src['model_bin_sha256']})")
os.environ["HF_HUB_OFFLINE"] = "1"   # từ đây không tải gì thêm từ Hub: mô hình nào chưa ghim sẽ báo lỗi

In [ ]:
# Ô 6. Hàm phụ: chạy lệnh có log + đo thời gian, đồng bộ lên Drive, kiểm một lượt chạy
def now():
    return time.strftime("%Y-%m-%d %H:%M:%S")

def run(args, log, run_dir=None, step=None, asr_step=None):
    # Chạy trong tts_gen/, in và ghi log (nối thêm). step -> run_config.json["colab_steps"]; asr_step -> logs/thoi_gian_asr.json
    log = log if log.startswith("/") else f"{TG}/{log}"
    os.makedirs(os.path.dirname(log), exist_ok=True)
    env = dict(os.environ, PYTHONUNBUFFERED="1", HF_HUB_OFFLINE="1")
    t0 = time.time()
    with open(log, "a", encoding="utf-8") as f:
        f.write(f"\n===== {now()} {' '.join(map(str, args))}\n")
        p = subprocess.Popen(list(map(str, args)), cwd=TG, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, env=env)
        for line in p.stdout:
            f.write(line); f.flush()
            if "Xet Storage" not in line and "hf_xet" not in line:
                print(line, end="")
        rc = p.wait()
    dt = round(time.time() - t0, 1)
    if run_dir and step:
        cp = f"{TG}/{run_dir}/run_config.json"
        cfg = json.load(open(cp, encoding="utf-8")) if os.path.exists(cp) else {}
        cfg.setdefault("colab_steps", []).append(dict(buoc=step, giay=dt, rc=rc, luc=now()))
        cfg["colab_env"] = json.load(open(TG + "/runs/colab_env.json", encoding="utf-8"))
        json.dump(cfg, open(cp, "w", encoding="utf-8"), ensure_ascii=False, indent=1)
    if run_dir and asr_step:
        tp = f"{TG}/{run_dir}/logs/thoi_gian_asr.json"
        os.makedirs(os.path.dirname(tp), exist_ok=True)
        t = json.load(open(tp, encoding="utf-8")) if os.path.exists(tp) else {}
        t[asr_step] = round(t.get(asr_step, 0) + dt, 1)
        json.dump(t, open(tp, "w", encoding="utf-8"), ensure_ascii=False, indent=1)
    print(f"[{step or asr_step or args[1]}] rc={rc}, {dt} s")
    return rc

def sync(run_dir):
    # Nén <run_dir> (tương đối với tts_gen/) và chép lên Drive (ghi tạm rồi đổi tên)
    name = run_dir.replace("/", "__")
    os.makedirs("/content/sync", exist_ok=True)
    local = f"/content/sync/{name}.tar.gz"
    with tarfile.open(local, "w:gz") as t:
        t.add(f"{TG}/{run_dir}", arcname=run_dir)
    shutil.copy(local, f"{OUT_DRIVE}/{name}.tar.gz.tmp")
    os.replace(f"{OUT_DRIVE}/{name}.tar.gz.tmp", f"{OUT_DRIVE}/{name}.tar.gz")
    print(f"Đã chép {run_dir} lên Drive ({os.path.getsize(local) / 2**20:.1f} MB)")

def restore(run_dir):
    # Máy ảo mới (không còn /content): lấy bản đã chép lên Drive để chạy tiếp
    if os.path.exists(f"{TG}/{run_dir}/manifest.jsonl"):
        return
    p = f"{OUT_DRIVE}/{run_dir.replace('/', '__')}.tar.gz"
    if os.path.exists(p):
        with tarfile.open(p) as t:
            t.extractall(TG)
        print("Khôi phục", run_dir, "từ Drive")

def jl(p):
    p = p if p.startswith("/") else f"{TG}/{p}"
    return [json.loads(l) for l in open(p, encoding="utf-8") if l.strip()]

def asr_done(run_dir, tag):
    p = f"{TG}/{run_dir}/asr_{tag}.jsonl"
    if not os.path.exists(p):
        return False
    return {m["utt_id"] for m in jl(f"{run_dir}/manifest.jsonl")} <= {d["utt_id"] for d in jl(p)}

def check_run(run_dir, plan):
    # Kiểm: đủ sample_id, không trùng utt_id, đủ tệp âm thanh, tts_text được dùng, tỷ lệ sinh lại
    sys.path.insert(0, TG)
    from eval_common import active_rows
    man = jl(f"{run_dir}/manifest.jsonl"); act = active_rows(man)
    want = {r["sample_id"]: r for r in jl(plan)}
    utt = [m["utt_id"] for m in man]
    missing_files = [m["utt_id"] for m in man for k in ("audio48", "audio16") if not os.path.exists(f"{TG}/{run_dir}/{m[k]}")]
    tts_bad = [m["sample_id"] for m in man if want.get(m["sample_id"], {}).get("tts_text") and m.get("tts_input") != want[m["sample_id"]]["tts_text"]]
    retried = sum(bool(m.get("retries")) for m in act)
    res = dict(dong_manifest=len(man), sample_id_duy_nhat=len({m["sample_id"] for m in act}), ke_hoach=len(want),
               thieu=sorted(set(want) - {m["sample_id"] for m in act})[:10], thua=sorted({m["sample_id"] for m in act} - set(want))[:10],
               utt_id_trung=len(utt) - len(set(utt)), thieu_tep=missing_files[:10], tts_text_khong_dung=tts_bad[:10],
               co_sinh_lai=retried, ty_le_sinh_lai=round(retried / max(len(act), 1), 3),
               con_loi_ky_thuat=sum(m.get("tts_check", "dat") != "dat" for m in act),
               giong_bi_loai=sorted({m["voice"] for m in man if m["voice"] == "Minh Triết"}))
    print(json.dumps(res, ensure_ascii=False, indent=1))
    dung = []
    if res["thieu"] or res["thua"] or res["utt_id_trung"] or res["thieu_tep"]:
        dung.append("mất hoặc trùng câu / thiếu tệp")
    if res["tts_text_khong_dung"]:
        dung.append("tts_text không được dùng")
    if res["ty_le_sinh_lai"] > 0.25:
        dung.append("quá 25% số câu phải sinh lại")
    if dung:
        print("*** ĐIỂM DỪNG:", "; ".join(dung), "-> dừng và báo, không chạy tiếp ***")
    return res, dung

DT_ARGS = ["--dtype", DTYPE, *TTS_PIN_ARGS] if DTYPE else None
print("Đã nạp hàm phụ. DTYPE =", DTYPE)

## Bước kiểm kiểu số (T4 không có bf16)

Sinh lại 96 câu thí điểm (bỏ 4 câu của Minh Triết), **cùng giọng** như `runs/pilot100` (bf16 trên laptop), lô 16,
`--max-retry 0` vì lượt bf16 trên laptop không sinh lại câu nào. Chấm bằng ASR: medium lô 2, large CT2 lô 1.

In [ ]:
# Ô 7. Sinh pilot100 bằng fp16 và fp32
for dt, name in (("float16", "runs/pilot100_t4_fp16"), ("float32", "runs/pilot100_t4_fp32")):
    restore(name)
    run([PY, "run_pilot.py", "--input", "pilot/pilot100_voices.jsonl", "--out", name, "--batch", 16, "--max-retry", 0,
         "--dtype", dt, *TTS_PIN_ARGS], f"{name}/logs/tts.log", name, f"tts_{dt}")
    sync(name)

In [ ]:
# Ô 8. ASR cho hai lượt: medium lô 2, rồi large CT2 lô 1 (không chạy cùng lúc)
for name in ("runs/pilot100_t4_fp16", "runs/pilot100_t4_fp32"):
    if not asr_done(name, "PhoWhisper-medium"):
        run([PY, "asr_check.py", name, "--model", PIN["PhoWhisper-medium"], "--batch", 2, "--lexicon", "lexicon/asr_bien_the.tsv"],
            f"{name}/logs/asr.log", name, asr_step="asr_medium")
    if not asr_done(name, "PhoWhisper-large-ct2-int8"):
        run([PY, "asr_check.py", name, "--model", "vinai/PhoWhisper-large", "--backend", "ct2", "--ct2-dir", CT2_DIR,
             "--lexicon", "lexicon/asr_bien_the.tsv"], f"{name}/logs/asr.log", name, asr_step="asr_large_ct2")
    sync(name)

In [ ]:
# Ô 9. Bảng so với bf16 trên laptop -> DỪNG, gửi bảng, chờ chọn fp16 hay fp32
run([PY, "compare_dtype.py", "bf16_laptop=runs/pilot100", "fp16_t4=runs/pilot100_t4_fp16", "fp32_t4=runs/pilot100_t4_fp32",
     "--lexicon", "lexicon/asr_bien_the.tsv", "--out", "runs/so_sanh_kieu_so.md"], "runs/so_sanh_kieu_so.log")
for name in ("runs/pilot100", "runs/pilot100_t4_fp16", "runs/pilot100_t4_fp32"):
    c = json.load(open(f"{TG}/{name}/run_config.json", encoding="utf-8"))
    print(f"{name}: dtype {c.get('dtype')} | RTF {c.get('rtf_overall')} | nạp {c.get('load_s_total', c.get('load_s'))} s | "
          f"VRAM đỉnh {c.get('vram_peak_alloc_gb')} GB | GPU {c.get('gpu')} | thời gian ASR {json.load(open(f'{TG}/{name}/logs/thoi_gian_asr.json')) if os.path.exists(f'{TG}/{name}/logs/thoi_gian_asr.json') else '-'}")
shutil.copy(TG + "/runs/so_sanh_kieu_so.md", OUT_DRIVE + "/so_sanh_kieu_so.md")
print("\n==> DỪNG Ở ĐÂY: gửi bảng trên, chờ chọn DTYPE (ô 1) rồi mới chạy tiếp từ ô 10.")

## Sau khi đã chọn kiểu số (điền `DTYPE` ở ô 1, chạy lại ô 1 và ô 6)

In [ ]:
# Ô 10. a) Chạy thử 24 câu (mỗi giọng 1 câu; 19 câu có tts_text)
assert DTYPE in ("float16", "float32"), "Chưa chọn DTYPE ở ô 1"
name = "runs/eval1000_smoke"
restore(name)
run([PY, "run_pilot.py", "--input", "runs/eval1000_A/logs/input_smoke24.jsonl", "--out", name, "--batch", 16, *DT_ARGS],
    f"{name}/logs/tts.log", name, "tts_smoke")
res, dung = check_run(name, "runs/eval1000_A/logs/input_smoke24.jsonl")
man = jl(f"{name}/manifest.jsonl")
need = ["tts_text", "tts_input", "loan_arm", "eval_idx", "shard", "giong_khop_tieu_vung", "voice_style"]
print("Thiếu trường mới:", [k for k in need if any(k not in m for m in man)] or "không",
      "| câu có tts_text:", sum(bool(m.get("tts_text")) for m in man), "| dtype:", json.load(open(f"{TG}/{name}/run_config.json"))["dtype"])
sync(name)

In [ ]:
# Ô 11. b) 4 câu Thái Sơn: cấu hình mặc định và --no-ref-codes (cấu hình B); sinh lại tối đa 2 lần như lượt chính
for name, extra in (("runs/thai_son_mac_dinh", []), ("runs/thai_son_no_ref", ["--no-ref-codes"])):
    restore(name)
    run([PY, "run_pilot.py", "--input", "runs/eval1000_A/logs/input_thai_son4.jsonl", "--out", name, "--batch", 16,
         *DT_ARGS, *extra], f"{name}/logs/tts.log", name, "tts_thai_son")
    for m in jl(f"{name}/manifest.jsonl"):
        print(f"  {name.split('/')[-1]:16s} {m['sample_id']} lần {m['attempt'] + 1}: {m['tts_check']} (đầu {m['head_db']} dB) | {m['text_spoken']}")
    sync(name)

## c) Sinh phần A và thử ngắt đột ngột

Chạy ô 12. **Khi log in xong khoảng 12 trong 24 giọng** (dòng dạng `Tên giọng a0 ... câu`): Runtime → **Restart session**
(tiến trình bị diệt như mất điện; `/content` vẫn còn). Sau đó chạy lại ô 1, 2, 3, 4, 5, 6 (tự bỏ qua phần đã làm) rồi chạy lại ô 12.
Lần chạy bị ngắt sẽ mất `run_config` của riêng lần đó (đã chấp nhận); ô 12 ghi số dòng manifest trước mỗi lần chạy vào
`logs/thu_ngat.jsonl` làm bằng chứng. Nếu máy ảo bị thu hồi hẳn, `restore()` lấy bản trên Drive (chỉ có sau khi ô 12 chạy xong một lần).

In [ ]:
# Ô 12. Sinh phần A vào runs/eval1000_A
assert DTYPE in ("float16", "float32"), "Chưa chọn DTYPE ở ô 1"
name = "runs/eval1000_A"
restore(name)
os.makedirs(f"{TG}/{name}/logs", exist_ok=True)
mp = f"{TG}/{name}/manifest.jsonl"
n_before = len(jl(mp)) if os.path.exists(mp) else 0
with open(f"{TG}/{name}/logs/thu_ngat.jsonl", "a", encoding="utf-8") as f:
    f.write(json.dumps(dict(luc=now(), su_kien="bat_dau_chay", dong_manifest=n_before)) + "\n")
rc = run([PY, "run_pilot.py", "--input", "eval1000_tts/shard_A.jsonl", "--out", name, "--batch", 16, *DT_ARGS],
         f"{name}/logs/tts_A.log", name, "tts_A")
with open(f"{TG}/{name}/logs/thu_ngat.jsonl", "a", encoding="utf-8") as f:
    f.write(json.dumps(dict(luc=now(), su_kien="xong", rc=rc, dong_manifest=len(jl(mp)))) + "\n")
print(open(f"{TG}/{name}/logs/thu_ngat.jsonl").read())
res, dung = check_run(name, "eval1000_tts/shard_A.jsonl")
json.dump(dict(res, diem_dung=dung), open(f"{TG}/{name}/logs/kiem_tra_A_sau_tts.json", "w"), ensure_ascii=False, indent=1)
sync(name)

In [ ]:
# Ô 13. d) Đổi giọng cho câu vẫn lỗi kỹ thuật sau 2 lần sinh lại (chỉ 1 vòng)
name = "runs/eval1000_A"
fb = f"{TG}/{name}/fallback_input.jsonl"
if not os.path.exists(fb):
    run([PY, "make_fallback.py", name, "--run-tts", "--batch", 16, "--pilot-args", " ".join(DT_ARGS)],
        f"{name}/logs/fallback.log", name, "fallback")
else:   # đã có tệp (lần chạy trước bị ngắt): chạy tiếp run_pilot với chính tệp đó
    run([PY, "run_pilot.py", "--input", fb, "--out", name, "--batch", 16, *DT_ARGS], f"{name}/logs/fallback.log", name, "fallback")
res, dung = check_run(name, "eval1000_tts/shard_A.jsonl")
json.dump(dict(res, diem_dung=dung), open(f"{TG}/{name}/logs/kiem_tra_A_sau_doi_giong.json", "w"), ensure_ascii=False, indent=1)
sync(name)

In [ ]:
# Ô 14. e) ASR theo thứ tự: medium lô 2 -> large CT2 lô 1 -> asr_diagnose -> gate_union -> report_eval
name = "runs/eval1000_A"
if not asr_done(name, "PhoWhisper-medium"):
    run([PY, "asr_check.py", name, "--model", PIN["PhoWhisper-medium"], "--batch", 2, "--lexicon", "lexicon/asr_bien_the.tsv"],
        f"{name}/logs/asr_medium.log", name, asr_step="asr_medium")
    sync(name)
if not asr_done(name, "PhoWhisper-large-ct2-int8"):
    run([PY, "asr_check.py", name, "--model", "vinai/PhoWhisper-large", "--backend", "ct2", "--ct2-dir", CT2_DIR,
         "--lexicon", "lexicon/asr_bien_the.tsv"], f"{name}/logs/asr_large.log", name, asr_step="asr_large_ct2")
    sync(name)
if not os.path.exists(f"{TG}/{name}/asr_diagnose.csv"):
    run([PY, "asr_diagnose.py", name, "--whisper", PIN["PhoWhisper-medium"], "--ctc", PIN["wav2vec2"]],
        f"{name}/logs/asr_diagnose.log", name, asr_step="asr_diagnose")
    sync(name)
run([PY, "gate_union.py", name, "--lexicon", "lexicon/asr_bien_the.tsv"], f"{name}/logs/gate_union.log", name, asr_step="gate_union")
run([PY, "report_eval.py", name, "--plan", "eval1000_tts/eval_1000.jsonl", "--out", "reports/eval1000"],
    f"{name}/logs/report_eval.log", name, step="report_eval")
sync(name); sync("reports/eval1000")

In [ ]:
# Ô 15. Đóng gói kết quả để tải về laptop: manifest, run_config, asr_*, gate_hop, báo cáo, log, audio48 phần A
out = "/content/ket_qua_A.tar.gz"
with tarfile.open(out, "w:gz") as t:
    A = f"{TG}/runs/eval1000_A"
    for p in sorted(glob.glob(A + "/*")):
        if os.path.basename(p) != "audio16":
            t.add(p, arcname="runs/eval1000_A/" + os.path.basename(p))
    for p in ["reports/eval1000", "runs/so_sanh_kieu_so.md", "runs/colab_env.json"]:
        if os.path.exists(f"{TG}/{p}"):
            t.add(f"{TG}/{p}", arcname=p)
    for run_dir in ["runs/eval1000_smoke", "runs/thai_son_mac_dinh", "runs/thai_son_no_ref",
                    "runs/pilot100_t4_fp16", "runs/pilot100_t4_fp32"]:
        for p in sorted(glob.glob(f"{TG}/{run_dir}/*")):
            if os.path.basename(p) != "audio16" and not (run_dir.startswith("runs/pilot100_t4") and os.path.basename(p) == "audio48"):
                t.add(p, arcname=f"{run_dir}/{os.path.basename(p)}")
h = sha256_file(out)
shutil.copy(out, OUT_DRIVE + "/ket_qua_A.tar.gz")
open(OUT_DRIVE + "/ket_qua_A.tar.gz.sha256", "w").write(f"{h}  ket_qua_A.tar.gz\n")
print(f"{os.path.getsize(out) / 2**20:.1f} MB -> {OUT_DRIVE}/ket_qua_A.tar.gz | sha256 {h}")